# A3-M1 · 실습 과제 — 정답
준비 셀부터 순서대로 실행합니다. 02 노트북을 먼저 실행해 두면 좋습니다.

## 준비
모든 과제가 함께 쓰는 준비 셀입니다.

In [2]:
import re, numpy as np, tiktoken
from kiwipiepy import Kiwi
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers, processors
from transformers import PreTrainedTokenizerFast
from sklearn.feature_extraction.text import TfidfVectorizer
from shop_faq import FAQ, TESTS
kiwi = Kiwi()
o200k, cl100k = tiktoken.get_encoding("o200k_base"), tiktoken.get_encoding("cl100k_base")
corpus = open("shop_corpus.txt", encoding="utf-8").read().splitlines()
docs = [f"{q} {a}" for _, _, q, a in FAQ]
ids = [i for i, *_ in FAQ]

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 과제 1 · kiwi 사용자 사전
`"무선이어폰 블루투스스피커 AS 기간"`을 kiwi로 분석해 (형태, 품사)를 출력하고, `무선이어폰`·`블루투스스피커`를 고유명사(NNP)로 등록한 뒤 다시 출력하세요.

In [3]:
s = "무선이어폰 블루투스스피커 AS 기간"
print("전:", [(t.form, t.tag) for t in kiwi.tokenize(s)])
for w in ["무선이어폰", "블루투스스피커"]:
    kiwi.add_user_word(w, "NNP")
print("후:", [(t.form, t.tag) for t in kiwi.tokenize(s)])

전: [('무선', 'NNG'), ('이어폰', 'NNG'), ('블루투스', 'NNP'), ('스피커', 'NNG'), ('AS', 'SL'), ('기간', 'NNG')]
후: [('무선이어폰', 'NNP'), ('블루투스스피커', 'NNP'), ('AS', 'SL'), ('기간', 'NNG')]


## 과제 2 · 우리 WordPiece 만들기
말뭉치(`corpus`)와 FAQ 문서(`docs`)를 함께 써서 vocab_size=600 WordPiece를 학습하고, `[CLS] A [SEP] B [SEP]` 후처리를 붙이세요. `"무선이어폰 배송 언제 와요?"`와 `"평일 2~3일 걸립니다"`를 문장 쌍으로 encode해 tokens와 type_ids를 출력하세요.

In [4]:
tk = Tokenizer(models.WordPiece(unk_token="[UNK]"))
tk.normalizer = normalizers.NFKC()
tk.pre_tokenizer = pre_tokenizers.BertPreTokenizer()
tk.train_from_iterator(corpus + docs, trainers.WordPieceTrainer(vocab_size=600, min_frequency=2,
                       special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]))
tk.post_processor = processors.TemplateProcessing(single="[CLS] $A [SEP]", pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[("[CLS]", tk.token_to_id("[CLS]")), ("[SEP]", tk.token_to_id("[SEP]"))])
e = tk.encode("무선이어폰 배송 언제 와요?", "평일 2~3일 걸립니다")
print(e.tokens)
print(e.type_ids)

['[CLS]', '무', '##선', '##이', '##어', '##폰', '배송', '언제', '[UNK]', '?', '[SEP]', '평', '##일', '2', '~', '3', '##일', '걸', '##립', '##니다', '[SEP]']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## 과제 3 · 겹쳐 나누기
과제 2의 토크나이저를 `PreTrainedTokenizerFast`로 감싸고, 아래 `doc`을 max_length=20, stride=5로 나눠 조각마다 원문 범위를 출력하세요.

In [5]:
doc = "반품은 상품을 받은 날부터 7일 이내에 신청합니다. 단순 변심 반품은 왕복 배송비 6000원을 고객이 부담합니다. 파손 상품은 사진을 첨부하면 무료로 교환합니다."
hf = PreTrainedTokenizerFast(tokenizer_object=tk, unk_token="[UNK]", pad_token="[PAD]", cls_token="[CLS]", sep_token="[SEP]")
ch = hf(doc, truncation=True, max_length=20, stride=5, return_overflowing_tokens=True, return_offsets_mapping=True)
for i, offs in enumerate(ch["offset_mapping"]):
    real = [o for o in offs if o != (0, 0)]
    print(f"조각 {i}: {doc[real[0][0]:real[-1][1]]}")

조각 0: 반품은 상품을 받은 날부터 7일 이내에 신청합니다. 단순 변
조각 1: 합니다. 단순 변심 반품은 왕복 배송비 6000원을 고객
조각 2: 0원을 고객이 부담합니다. 파손 상품은 사진을 첨부하면
조각 3: 은 사진을 첨부하면 무료로 교환합니다.


## 과제 4 · 인코딩별 토큰 수 표
평가 질문 12개(`TESTS`)의 글자 수, cl100k 토큰 수, o200k 토큰 수를 표로 출력하고, 두 인코딩의 차이가 가장 큰 질문을 찾으세요.

In [6]:
rows = [(q, len(q), len(cl100k.encode(q)), len(o200k.encode(q))) for q, _ in TESTS]
for q, n, a, b in rows:
    print(f"{q:<24} 글자 {n:>2}  cl100k {a:>2}  o200k {b:>2}")
best = max(rows, key=lambda r: r[2] - r[3])
print("차이가 가장 큰 질문:", best[0], f"({best[2]} → {best[3]})")
print("합계: cl100k", sum(r[2] for r in rows), "/ o200k", sum(r[3] for r in rows))

택배 언제 와요?                글자  9  cl100k 11  o200k  6
배송비 무료 기준이 뭐예요?          글자 15  cl100k 18  o200k  9
내 택배 지금 어디쯤인지 보고 싶어요     글자 20  cl100k 22  o200k 13
소파 같은 큰 물건은 어떻게 와요?      글자 19  cl100k 23  o200k 11
미국으로 보내줄 수 있나요?          글자 15  cl100k 13  o200k  9
카드 할부 몇 개월까지 돼요?         글자 16  cl100k 17  o200k 12
받은 지 5일 됐는데 반품 돼요?       글자 18  cl100k 20  o200k 15
물건이 깨져서 왔어요              글자 11  cl100k 15  o200k  9
환불 며칠 걸려요?               글자 10  cl100k 13  o200k  9
주문 취소 어떻게 해요?            글자 13  cl100k 14  o200k  8
비번 까먹었어요                 글자  8  cl100k 12  o200k  6
포인트 유효기간 있나요?            글자 13  cl100k 15  o200k  8
차이가 가장 큰 질문: 소파 같은 큰 물건은 어떻게 와요? (23 → 11)
합계: cl100k 193 / o200k 115


## 과제 5 · 품사 하나가 검색을 바꾼다
kiwi 토크나이저의 KEEP 품사 집합에 대명사(`NP`)와 일반부사(`MAG`)를 추가했을 때 TF-IDF 검색 Hit@1이 어떻게 바뀌는지 비교하세요.

In [7]:
BASE = {"NNG", "NNP", "NNB", "SL", "SN", "VV", "VA", "XR"}
def hit1(keep):
    f = lambda s: [t.form for t in kiwi.tokenize(s) if t.tag in keep]
    vec = TfidfVectorizer(tokenizer=f, lowercase=False, token_pattern=None)
    X = vec.fit_transform(docs)
    return sum(ids[(X @ vec.transform([q]).T).toarray().ravel().argmax()] == g for q, g in TESTS)
print("기본      :", hit1(BASE), "/ 12")
print("+NP +MAG  :", hit1(BASE | {"NP", "MAG"}), "/ 12")

기본      : 7 / 12
+NP +MAG  : 8 / 12


## 과제 6 · 두 토크나이저 점수 섞기
kiwi 형태소 TF-IDF 점수와 글자 2-gram TF-IDF 점수를 같은 비율(0.5:0.5)로 더해 검색하면 Hit@1이 몇이 되는지 구하세요.

In [8]:
KEEP = {"NNG", "NNP", "NNB", "SL", "SN", "VV", "VA", "XR"}
fk = lambda s: [t.form for t in kiwi.tokenize(s) if t.tag in KEEP]
fc = lambda s: [s[i:i+2] for i in range(len(s) - 1) if " " not in s[i:i+2]]
vk = TfidfVectorizer(tokenizer=fk, lowercase=False, token_pattern=None); Xk = vk.fit_transform(docs)
vc = TfidfVectorizer(tokenizer=fc, lowercase=False, token_pattern=None); Xc = vc.fit_transform(docs)
hit = 0
for q, g in TESTS:
    s = 0.5 * (Xk @ vk.transform([q]).T).toarray().ravel() + 0.5 * (Xc @ vc.transform([q]).T).toarray().ravel()
    hit += ids[s.argmax()] == g
print("섞은 점수 Hit@1:", hit, "/ 12")

섞은 점수 Hit@1: 9 / 12
